<img src="https://cdn.jsdelivr.net/gh/aureliensalas/python_finance@main/ressources/img/logo_macmia.png" alt="Banque des Territoires · France 2030 · MACMIA" width="520">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aureliensalas/python_finance/blob/main/ml_crypto/cours/seance2_cours.ipynb)

# Séance 8 : prévoir une décision

**Cours** · 2h · machine learning, seconde partie

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/aureliensalas/python_finance/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- dire ce qu'est une classification, et pourquoi une droite ne suffit plus
- lire ce que sort un modèle de classification : une probabilité, pas une décision
- transformer une probabilité en décision, et nommer les deux façons de se tromper
- lire une matrice de confusion, et en tirer justesse, précision et rappel
- choisir un seuil en fonction de ce que coûte chaque erreur

## La phrase du bloc

> **Un modèle n'a pas de valeur parce qu'il colle aux données. Il en a s'il tient sur des données qu'il n'a jamais vues.**

Et celle de la séance, qu'on ne comprendra vraiment qu'à la fin :

> **Le modèle donne une probabilité. La décision, c'est nous.**

> **Les cellules « Prédire ».** Avant de les exécuter, on annonce **à voix haute** la valeur attendue. Rien à écrire : on dit, on exécute, on compare.

Exécutez d'abord la cellule de setup.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score

pd.set_option("display.max_rows", 12)
BASE = "https://cdn.jsdelivr.net/gh/aureliensalas/python_finance@main/pandas_crypto/data/"


def verifier(nom, condition, indice=""):
    if condition:
        print("OK       -", nom)
    else:
        print("A REVOIR -", nom, ":", indice)

Une fonction de mise en page : `afficher_matrice`, qui donne des noms aux cases d'une matrice de confusion (section 4). **Vous n'avez pas à la lire.**

In [ ]:
def afficher_matrice(matrice, non="non", oui="oui"):
    """La matrice de confusion, avec le nom de chaque ligne et de chaque colonne."""
    return pd.DataFrame(matrice,
                        index=["réel : " + non, "réel : " + oui],
                        columns=["prévu : " + non, "prévu : " + oui])

## 0. Échauffement

On repart de la prévision qui marchait : le volume du bitcoin. La table d'abord.

In [ ]:
crypto = pd.read_csv(BASE + "crypto.csv")
crypto["date"] = pd.to_datetime(crypto["date"])
btc = crypto.query("coin == 'BTC'").sort_values("date").reset_index(drop=True)
btc["r"] = btc["close"].pct_change() * 100
btc.head()

### Écrire

Les quatre colonnes de la séance 7 : `vol`, le montant échangé en milliards de dollars ; `m5` et `m20`, ses moyennes sur 5 et 20 jours ; `vol_demain`, le volume du lendemain.

In [ ]:
_v = btc["volume"] * btc["close"] / 1e9
verifier("vol", "vol" in btc.columns and abs(btc["vol"].median() - _v.median()) < 1e-6, "volume × close, divisé par 1e9")
verifier("m5 et m20", "m20" in btc.columns and abs(btc["m20"][30] - _v.rolling(20).mean()[30]) < 1e-6 and abs(btc["m5"][30] - _v.rolling(5).mean()[30]) < 1e-6, "rolling(5).mean() et rolling(20).mean() sur vol")
verifier("vol_demain", "vol_demain" in btc.columns and abs(btc["vol_demain"][0] - _v[1]) < 1e-6, "shift(-1) : la valeur de demain sur la ligne d'aujourd'hui")

Deux colonnes de plus : l'**amplitude** du jour — la taille du mouvement, sans son signe, avec `abs` — et le rendement du lendemain, pour la fin de la séance. Puis on retire les lignes incomplètes.

In [ ]:
btc["ampl"] = btc["r"].abs()
btc["r_demain"] = btc["r"].shift(-1)
v = btc.dropna(subset=["m20", "vol_demain"]).reset_index(drop=True)
print(len(v))

## 1. D'un nombre à une catégorie

À la séance 7, on prévoyait **combien**. Aujourd'hui, une question plus simple et plus fréquente : **oui ou non**.

> **Demain sera-t-il une journée de gros volume** — un volume au-dessus de sa moyenne des 20 derniers jours ?

La réponse n'est plus un nombre, c'est une **catégorie**. C'est une **classification**, le second cas du supervisé. La plupart des décisions en entreprise ont cette forme : accorder le prêt ou non, bloquer la carte ou non, renforcer les serveurs ou non.

La cible, c'est une comparaison sur deux colonnes — le geste du bloc 2. `astype(int)` transforme `True` en 1 et `False` en 0.

In [ ]:
v["gros"] = (v["vol_demain"] > v["m20"]).astype(int)
v[["date", "vol", "m20", "vol_demain", "gros"]].head()

### Prédire

In [ ]:
print(v["gros"].mean())

Un jour sur deux, à peu près. On coupe, exactement comme à la séance 7. Les variables : ce qu'on sait le soir à minuit.

In [ ]:
X = v[["vol", "m5", "m20", "ampl"]]
y = v["gros"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
print(len(X_train), len(X_test))

### Pourquoi pas la droite de la séance 7 ?

On essaie. C'est une ligne.

In [ ]:
droite = LinearRegression().fit(X_train, y_train)
prevu_droite = droite.predict(X_test)
print(prevu_droite.min(), prevu_droite.max())
print((prevu_droite < 0).mean() + (prevu_droite > 1).mean())

La droite répond de **−0,34 à 2,16**, et **une prévision sur dix** sort de l'intervalle entre 0 et 1.

« 216 % de chances d'une journée de gros volume » : ça ne veut rien dire. Une droite monte sans fin ; une probabilité, elle, s'arrête à 1.

### La courbe en S

On garde l'idée de la droite, et on la fait passer dans une fonction qui l'**écrase entre 0 et 1** : la **fonction logistique**. Les deux, côte à côte :

In [ ]:
z = pd.Series(np.linspace(-6, 6, 121))
courbes = pd.DataFrame({"z": z, "une droite": 0.5 + z / 8, "la courbe logistique": 1 / (1 + np.exp(-z))})
courbes.plot(x="z", title="Une droite ne s'arrête pas ; la logistique reste entre 0 et 1", figsize=(8, 4))

Loin à gauche, la courbe colle à 0. Loin à droite, à 1. Au milieu, elle monte. Ce qui en sort **se lit comme une probabilité**.

### Ce que fait `fit`, statistiquement

Pour la droite, c'était : choisir la droite qui rend **la somme des erreurs au carré** la plus petite. Pour la logistique, c'est : choisir la courbe qui rend **les réponses observées les plus probables**. Si les jours de gros volume ont reçu une probabilité haute, et les autres une probabilité basse, la courbe est bonne. Cette méthode s'appelle le **maximum de vraisemblance** : le nom, et on n'y revient pas.

L'outil s'appelle `LogisticRegression` — « régression » pour des raisons historiques, mais c'est bien une classification. Même forme qu'avant.

### Écrire

`modele`, une `LogisticRegression` ajustée sur l'entraînement. Mettez `max_iter=1000` entre les parenthèses : c'est le nombre d'essais qu'on lui autorise pour trouver la courbe ; sans lui, il peut s'arrêter trop tôt et afficher un avertissement.

In [ ]:
verifier("modele", hasattr(modele, "coef_") and modele.coef_.shape == (1, 4), "LogisticRegression(max_iter=1000).fit(X_train, y_train)")

## 2. Ce que sort le modèle : `predict_proba`

Une méthode nouvelle. Regardez ce qu'elle donne pour les cinq premiers soirs du test.

In [ ]:
modele.predict_proba(X_test)[:5].round(3)

**Deux colonnes par soir** : la probabilité de « non », puis celle de « oui ». Elles font 1 à elles deux. On garde la seconde, celle de « gros volume ».

Pour la prendre : `[:, 1]`. Les deux-points veulent dire « toutes les lignes », le 1 « la colonne numéro 1 » — la seconde, puisqu'on compte à partir de 0 depuis le bloc 1.

### Écrire

`proba`, la probabilité de « gros volume » pour chaque soir du test.

In [ ]:
verifier("une probabilité par soir", len(proba) == len(y_test), "predict_proba sur X_test")
verifier("la bonne colonne", abs(proba - modele.predict_proba(X_test)[:, 1]).max() < 1e-12, "[:, 1] : la colonne de « oui » ; [:, 0] serait celle de « non »")

In [ ]:
pd.Series(proba).plot(kind="hist", bins=10, title="Ce que dit le modèle, soir après soir", xlabel="probabilité de gros volume demain", figsize=(8, 4))

**Ce que ça représente — on s'arrête dessus.** Pour chaque soir du test, le modèle dit par exemple : *« demain, il y a 83 % de chances que le volume dépasse sa moyenne »*. Ce n'est **pas** une décision. C'est un **degré de confiance**.

Et il a des choses à dire : les probabilités s'étalent de 0 à 1, avec beaucoup de soirs tranchés dans un sens ou dans l'autre.

> **Retenez ceci, on s'en servira à la fin.** Le modèle s'arrête là. Pour un même soir, il sort toujours la même probabilité. **Tout ce qui suit — dire oui ou non — c'est nous qui le décidons.**

### Prédire

In [ ]:
print((proba > 0.9).sum())

In [ ]:
print((proba < 0.1).sum())

## 3. Décider : pourquoi 50 %

**Une question pour vous.** Je vous pose une question, sans vous dire laquelle. Deux réponses possibles, A et B, sans vous dire ce qu'elles sont. Un algorithme vous souffle : *A a 51 % de chances d'être la bonne réponse, B en a 49 %*.

**Vous répondez quoi ?**

A, évidemment. Quand rien d'autre n'est en jeu, on choisit **le plus probable**. C'est le seuil de 50 % : on dit « oui » au-dessus, « non » en dessous.

In [ ]:
decision = (proba >= 0.5).astype(int)
decision[:10]

`modele.predict` fait exactement ça, en une ligne : il décide au seuil de 0,5.

In [ ]:
print((decision == modele.predict(X_test)).mean())

On vient de **transformer une probabilité en décision**. Et une décision, ça peut être faux.

## 4. Deux façons d'avoir raison, deux façons de se tromper

Chaque soir, on a dit **oui** ou **non**. Et le lendemain a été **gros** ou **normal**. Ça fait quatre cas, et chacun a un nom.

| | le lendemain a été **gros** | le lendemain a été **normal** |
|---|---|---|
| on a dit **oui** | **vrai positif** : l'alerte était justifiée | **faux positif** : une fausse alerte |
| on a dit **non** | **faux négatif** : on l'a raté | **vrai négatif** : le silence était justifié |

**Les deux erreurs ne sont pas la même erreur.** Pour chaque exemple, dites laquelle est la pire.

| | faux positif | faux négatif |
|---|---|---|
| **test COVID** | quelqu'un s'isole pour rien | un malade va dîner chez ses grands-parents |
| **test de grossesse** | une fausse joie, ou une fausse peur, corrigée à la prise de sang | on continue à boire et à fumer |
| **alarme incendie** | on sort sous la pluie pour rien | l'immeuble brûle sans que personne ne soit prévenu |

Dans les trois, le faux négatif est bien pire. Ce n'est pas toujours le cas — et c'est exactement ce qu'on va exploiter à la section 7.

### La matrice de confusion

Elle range les quatre cas dans un tableau. `confusion_matrix` compte, et `afficher_matrice` met les noms.

In [ ]:
matrice = confusion_matrix(y_test, decision)
afficher_matrice(matrice, non="normal", oui="gros volume")

On lit case par case, avec les noms du tableau :

- **264 vrais négatifs** : on a dit « normal », et c'était normal ;
- **81 faux positifs** : on a dit « gros », et c'était normal ;
- **107 faux négatifs** : on a dit « normal », et c'était gros — **ratés** ;
- **177 vrais positifs** : on a dit « gros », et c'était gros.

Pour récupérer les quatre nombres, `ravel` déplie la matrice ligne par ligne. L'ordre est donc toujours le même : vrais négatifs, faux positifs, faux négatifs, vrais positifs.

In [ ]:
vn, fp, fn, vp = matrice.ravel()
print(vn, fp, fn, vp)

## 5. La justesse, et le cas extrême

La première question qu'on se pose : **dans quelle part des cas a-t-on eu raison ?** Les deux cases où on a raison, sur le total. C'est la **justesse**.

Et contre quoi la comparer ? Contre la **règle paresseuse**, qui répond toujours « normal ». Elle a raison chaque fois que le lendemain est normal : sa justesse, c'est la part de « non ».

### Écrire

`justesse`, à partir de `vn`, `vp` et du nombre de soirs du test. Puis `paresseuse`, la justesse de la règle qui dit toujours « non ».

In [ ]:
verifier("justesse", abs(justesse - accuracy_score(y_test, decision)) < 1e-9, "les deux cases de la diagonale, vn + vp, divisées par len(y_test)")
verifier("paresseuse", abs(paresseuse - (1 - y_test.mean())) < 1e-9, "la part de « non » : 1 - y_test.mean()")

**70 % contre 55 %.** Le modèle fait quinze points de mieux que la paresse, sur une vraie prévision du lendemain. `accuracy_score(y_test, decision)` donne le même nombre en une ligne.

### Le cas extrême

On cherche maintenant les **journées exceptionnelles** : un volume de plus d'**une fois et demie** la moyenne des 20 jours. Même modèle, mêmes variables ; seule la cible change.

In [ ]:
v["exceptionnel"] = (v["vol_demain"] > 1.5 * v["m20"]).astype(int)
X_train, X_test, y_train_exc, y_test_exc = train_test_split(X, v["exceptionnel"], test_size=0.2, shuffle=False)
print(y_test_exc.sum(), len(y_test_exc))

67 journées exceptionnelles sur 629 : **une sur dix**.

### Écrire

`modele_exc`, ajusté sur cette cible ; `proba_exc`, ses probabilités sur le test ; `decision_exc` au seuil de 0,5 ; et `justesse_exc`.

In [ ]:
_m = LogisticRegression(max_iter=1000).fit(X_train, y_train_exc)
verifier("proba_exc", abs(proba_exc - _m.predict_proba(X_test)[:, 1]).max() < 1e-6, "fit sur X_train, y_train_exc ; predict_proba(X_test)[:, 1]")
verifier("decision_exc", (decision_exc == (proba_exc >= 0.5)).all(), "(proba_exc >= 0.5).astype(int)")
verifier("justesse_exc", abs(justesse_exc - accuracy_score(y_test_exc, decision_exc)) < 1e-9, "accuracy_score(y_test_exc, decision_exc)")

In [ ]:
print("règle paresseuse :", 1 - y_test_exc.mean())
matrice_exc = confusion_matrix(y_test_exc, decision_exc)
afficher_matrice(matrice_exc, non="normal", oui="exceptionnel")

**Justesse : 90,0 %. Règle paresseuse : 89,3 %.**

Et la matrice : le modèle a annoncé **6** des 67 journées exceptionnelles. **Il en a raté 61.** Et il affiche 90 % de justesse — parce que neuf jours sur dix sont normaux, et qu'il suffit de dire « normal » pour avoir raison neuf fois sur dix.

> **Une justesse seule ne veut rien dire.** Plus l'événement est rare, plus elle ment. On ne montre **jamais** une justesse sans la règle paresseuse à côté. Et il faut d'autres mesures.

## 6. Précision et rappel : l'alarme incendie

Le modèle est une **alarme incendie**. Deux questions, qui n'ont rien à voir l'une avec l'autre :

| | La question | Le calcul |
|---|---|---|
| **rappel** | De toutes les fois où il y a eu le feu, **combien de fois l'alarme a-t-elle sonné ?** | vrais positifs / tous les vrais feux |
| **précision** | De toutes les fois où l'alarme a sonné, **combien de fois y avait-il vraiment le feu ?** | vrais positifs / toutes les alarmes |

Sur la matrice : le rappel regarde **la ligne** des vrais feux ; la précision regarde **la colonne** des alarmes.

Une alarme qui sonne tout le temps a un rappel parfait et une précision nulle. Une alarme qui ne sonne qu'en voyant les flammes a une précision parfaite et rate la moitié des incendies. **On veut les deux, et on ne peut pas maximiser les deux à la fois.**

### Écrire

Pour les journées de gros volume, au seuil de 0,5 : `rappel` et `precision`, à la main, à partir de `vp`, `fp`, `fn`.

In [ ]:
verifier("rappel", abs(rappel - recall_score(y_test, decision)) < 1e-9, "les vrais feux sont sur la ligne du bas : vp + fn")
verifier("precision", abs(precision - precision_score(y_test, decision)) < 1e-9, "les alarmes sont dans la colonne de droite : vp + fp")

`recall_score` et `precision_score` donnent les mêmes nombres en une ligne. On s'en sert pour les journées exceptionnelles :

In [ ]:
print("rappel    :", recall_score(y_test_exc, decision_exc))
print("précision :", precision_score(y_test_exc, decision_exc))

| | gros volume, seuil 0,5 | journée exceptionnelle, seuil 0,5 |
|---|---|---|
| rappel | **62 %** | **9 %** |
| précision | **69 %** | **75 %** |

L'alarme des journées exceptionnelles est **timide**. Quand elle sonne, elle a raison trois fois sur quatre. Mais elle ne sonne presque jamais, et laisse passer **91 % des incendies**. C'est exactement ce que la justesse de 90 % cachait.

**Deux questions pour vous.** Pour une alarme incendie, lequel des deux voulez-vous haut ? Pour un filtre anti-spam, qui peut envoyer à la corbeille le mail d'un recruteur ?

<details><summary>Réponses</summary>

**L'alarme incendie : le rappel.** On veut attraper tous les feux, quitte à sortir sous la pluie pour rien.

**Le filtre anti-spam : la précision.** Ce qu'il met à la corbeille doit vraiment être du spam ; laisser passer quelques pubs est un moindre mal.
</details>

## 7. 0,5 n'a rien de sacré

### Retour à la question A ou B — avec un enjeu

Même question inconnue. Mêmes 51 % pour A, 49 % pour B. Mais cette fois :

> *Si vous répondez A et que c'est faux, il vous arrive quelque chose de très grave. Si vous répondez B et que c'est faux, il ne se passe rien.*

**Vous répondez quoi ?** B, évidemment. 51 % ne suffit plus.

Ce qui décide n'est donc pas seulement **ce qui est le plus probable**. C'est aussi **ce que coûte chaque erreur**. L'alarme incendie, c'est ça : on préfère sortir dix fois sous la pluie pour rien que rater un seul incendie. Une bonne alarme sonne dès qu'il y a une chance sérieuse de feu — à 10 %, pas à 50 %.

### Ce qui change, et ce qui ne change pas

C'est la phrase de la section 2. **Le modèle ne change pas** : mêmes variables, **mêmes probabilités**, soir après soir. Ce qui change, c'est **à partir de quelle probabilité nous agissons**. On baisse le seuil de 0,5 à 0,3, sur les journées exceptionnelles :

In [ ]:
decision_30 = (proba_exc >= 0.3).astype(int)
print("alertes à 0,5 :", decision_exc.sum(), "   alertes à 0,3 :", decision_30.sum())
afficher_matrice(confusion_matrix(y_test_exc, decision_30), non="normal", oui="exceptionnel")

### Pourquoi le rappel monte et la précision baisse

En passant de 0,5 à 0,3, les soirs dont la probabilité est **entre 0,3 et 0,5** passent du « non » au « oui ». Ce sont des soirs **moins sûrs** que ceux qui déclenchaient déjà l'alarme.

- Certains étaient de vrais feux qu'on ratait : on les attrape maintenant. **Le rappel monte** — et il ne peut que monter, puisqu'on n'enlève aucune alarme.
- Beaucoup étaient des soirs normaux : ce sont de nouvelles fausses alertes. Comme ces soirs sont moins sûrs, la part d'alarmes justifiées diminue. **La précision baisse.**

Baisser le seuil, c'est **échanger des faux négatifs contre des faux positifs**. Reste à savoir à quel prix.

### La situation qui donne un prix aux erreurs

Vous dirigez une **plateforme d'échange**. Les journées exceptionnelles saturent les serveurs : le 19 mai 2021, jour de krach, plusieurs grandes plateformes ont été ralenties ou coupées en pleine panique. Chaque soir, le modèle vous donne une probabilité, et vous décidez de **renforcer** ou non — serveurs supplémentaires, équipes de support mobilisées.

| erreur | ce qui se passe | coût |
|---|---|---|
| faux positif | on a renforcé pour rien | **20 000 €** |
| faux négatif | pas de renfort un jour exceptionnel : panne, frais perdus, clients partis | **100 000 €** |

### Écrire

La boucle est préparée : pour chaque seuil, elle refait la décision et la matrice. **Complétez les quatre `append`** : le nombre d'alertes (`vp_s + fp_s`), le rappel, la précision, et la facture en milliers d'euros (`fp_s * 20 + fn_s * 100`). C'est le motif « liste vide, `append` » du bloc 1.

In [ ]:
seuils = [0.5, 0.3, 0.2, 0.15, 0.1, 0.08, 0.05, 0.02]
alertes, rappels, precisions, factures = [], [], [], []
for s in seuils:
    decision_s = (proba_exc >= s).astype(int)
    vn_s, fp_s, fn_s, vp_s = confusion_matrix(y_test_exc, decision_s).ravel()
    # à vous : quatre lignes, une par liste

In [ ]:
_f = []
for _s in [0.5, 0.3, 0.2, 0.15, 0.1, 0.08, 0.05, 0.02]:
    _vn, _fp, _fn, _vp = confusion_matrix(y_test_exc, (proba_exc >= _s).astype(int)).ravel()
    _f.append(_fp * 20 + _fn * 100)
verifier("huit valeurs par liste", len(alertes) == len(rappels) == len(precisions) == len(factures) == 8, "un append par liste, dans la boucle")
verifier("les rappels", abs(rappels[0] - recall_score(y_test_exc, decision_exc)) < 1e-9, "vp_s / (vp_s + fn_s)")
verifier("les factures", list(factures) == _f, "fp_s * 20 + fn_s * 100, en milliers d'euros")

In [ ]:
tableau = pd.DataFrame({"seuil": seuils, "alertes": alertes, "rappel": rappels,
                        "précision": precisions, "facture (k€)": factures})
tableau.round(2)

In [ ]:
tableau.plot(x="seuil", y="facture (k€)", marker="o", title="La facture selon le seuil", ylabel="milliers d'euros", figsize=(8, 4))

Lisez le tableau de haut en bas : le rappel monte, la précision baisse, et la facture **descend, puis remonte**.

- À **0,5**, le seuil par défaut : **6 140 k€**. On rate presque toutes les journées exceptionnelles.
- Autour de **0,1** : **4 260 k€**. Un tiers de moins, **sans avoir touché au modèle**.
- En dessous, la facture remonte : à force d'alerter, les fausses alertes coûtent plus que ce qu'elles évitent.

Deux repères pour situer : ne jamais renforcer coûterait 6 700 k€ ; renforcer tous les soirs, 11 240 k€.

> **Le seuil n'est pas un réglage technique. C'est une décision de gestion.** Le creux dépend d'un rapport de coûts que seul le gestionnaire connaît. Aucune bibliothèque ne le trouvera à sa place.

<details><summary>Pour les curieux : où devrait tomber le creux ?</summary>

On renforce dès que le coût attendu d'un raté dépasse celui d'une fausse alerte : p × 100 > (1 − p) × 20, soit p > 20 / 120 ≈ **0,17**. On trouve le creux un peu plus bas, vers 0,1, parce que les journées exceptionnelles sont plus fréquentes dans le test (10,7 %) que dans l'entraînement (7,5 %) : le modèle, appris sur une période plus calme, sous-estime un peu les probabilités.
</details>

## 8. Et la direction de demain ?

Même méthode, une dernière cible : **demain sera-t-il une hausse ?** Variable : le rendement du jour.

In [ ]:
v["hausse"] = (v["r_demain"] > 0).astype(int)
Xd = v[["r"]]
Xd_train, Xd_test, yd_train, yd_test = train_test_split(Xd, v["hausse"], test_size=0.2, shuffle=False)
modele_dir = LogisticRegression(max_iter=1000).fit(Xd_train, yd_train)
proba_dir = modele_dir.predict_proba(Xd_test)[:, 1]

print("justesse :", accuracy_score(yd_test, (proba_dir >= 0.5).astype(int)))
print("part de hausses :", yd_test.mean())

**Justesse : 50,4 %, pour 49,3 % de journées de hausse.** Pile ou face.

### Prédire

La plus petite et la plus grande probabilité que le modèle ait données.

In [ ]:
print(proba_dir.min(), proba_dir.max())

In [ ]:
pd.Series(proba_dir).plot(kind="hist", bins=10, range=(0, 1), title="Ce que dit le modèle sur la direction", xlabel="probabilité de hausse demain", figsize=(8, 4))

Toutes les probabilités entre **0,40 et 0,64**. Le modèle ne sort jamais du « à peu près 50/50 ».

Remontez à l'histogramme de la section 2 : pour le volume, les probabilités s'étalaient de 0 à 1. **Pour le volume, le modèle a quelque chose à dire. Pour la direction, il hausse les épaules.** C'est la leçon de la séance 7, lue cette fois dans les probabilités : le volume a de la mémoire, le prix n'en a pas.

## 9. Ce que vous savez faire

| Vous voulez... | Vous écrivez |
|---|---|
| une cible oui / non | `(col_a > col_b).astype(int)` |
| ajuster une classification | `LogisticRegression(max_iter=1000).fit(X_train, y_train)` |
| les probabilités de « oui » | `modele.predict_proba(X_test)[:, 1]` |
| décider à un seuil | `(proba >= seuil).astype(int)` |
| la matrice de confusion | `confusion_matrix(y_test, decision)` |
| ses quatre cases | `vn, fp, fn, vp = matrice.ravel()` |
| la justesse, le rappel, la précision | `accuracy_score`, `recall_score`, `precision_score` |
| la règle paresseuse | `1 - y_test.mean()` |

## Cinq phrases à retenir

1. **Le modèle sort une probabilité ; la décision, c'est nous.** Même modèle, autre seuil, autres erreurs.
2. **Faux positif et faux négatif ne coûtent pas la même chose.** L'alarme incendie.
3. **Jamais une justesse seule.** La règle paresseuse à côté, toujours : 90 % de justesse peut vouloir dire 61 ratés sur 67.
4. **Rappel : de tous les feux, combien ai-je vus ? Précision : de toutes mes alarmes, combien étaient justifiées ?** Baisser le seuil monte le premier et baisse le second.
5. **Le seuil est un arbitrage de coûts**, pas un réglage technique.

## La suite

Deux travaux en autonomie ferment le bloc. **La fiche de desk** remobilise la régression : couvrir une position, prévoir la liquidité, prévoir l'agitation du marché. **Le banquier algorithmique** remobilise la classification sur de vrais dossiers de crédit — et se termine sur une question que le modèle ne tranche pas : a-t-on le droit, et a-t-on intérêt, à utiliser l'âge d'un client ?